# Simple Linear Regression Model

This notebook builds one Linear Regression model



## 1. Import libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.stats.outliers_influence import variance_inflation_factor


## 2. Load the dataset


In [ ]:
df = pd.read_csv("ecommerce_supervised_dataset.csv")

df.head()


## 3. Check dataset information


In [ ]:
print("Dataset shape:", df.shape)
print("\nMissing values:", df.isnull().sum().sum())

df.dtypes


## 4. Check correlation


In [ ]:
numerical_data = df.select_dtypes(
    include=["int64", "float64"]
)

correlation_matrix = numerical_data.corr()

correlation_matrix


## 5. Draw correlation heatmap


In [ ]:
plt.figure(figsize=(11, 8))

plt.imshow(
    correlation_matrix,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.colorbar(label="Correlation")

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=90
)

plt.yticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns
)

# Display correlation values inside the heatmap
for row in range(len(correlation_matrix.index)):
    for column in range(len(correlation_matrix.columns)):
        value = correlation_matrix.iloc[row, column]

        plt.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

plt.title("Correlation Heatmap with Values")
plt.tight_layout()
plt.show()


## 6. Separate input and target


In [ ]:
X = df.drop(
    columns=[
        "Purchase_Amount",
        "High_Value_Customer"
    ]
)

y = df["Purchase_Amount"]


## 7. Apply one-hot encoding


In [ ]:
X = pd.get_dummies(
    X,
    drop_first=True,
    dtype=int
)

X.head()


## 8. Check VIF


In [ ]:
vif = pd.DataFrame()

vif["Feature"] = X.columns

vif["VIF"] = [
    variance_inflation_factor(
        X.values,
        index
    )
    for index in range(X.shape[1])
]

vif.sort_values(
    "VIF",
    ascending=False
)


## 9. Split the data


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])


## 10. Create and train the model


In [ ]:
model = LinearRegression()

model.fit(
    X_train,
    y_train
)


## 11. Make predictions


In [ ]:
predictions = model.predict(
    X_test
)

predictions[:10]


## 12. Check model accuracy


In [ ]:
mae = mean_absolute_error(
    y_test,
    predictions
)

mse = mean_squared_error(
    y_test,
    predictions
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    predictions
)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)


## 13. Validate the model


In [ ]:
cv_scores = cross_val_score(
    LinearRegression(),
    X,
    y,
    cv=5,
    scoring="r2"
)

print("Cross-validation scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())


## 14. Compare actual and predicted values


In [ ]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions
})

results.head(10)


## 15. Draw actual versus predicted plot


In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    y_test,
    predictions
)

plt.xlabel("Actual Purchase Amount")
plt.ylabel("Predicted Purchase Amount")
plt.title("Actual vs Predicted")
plt.show()


## Final Results


In [ ]:
final_results = pd.DataFrame({
    "Metric": [
        "MAE",
        "MSE",
        "RMSE",
        "R²",
        "Mean CV R²"
    ],
    "Value": [
        mae,
        mse,
        rmse,
        r2,
        cv_scores.mean()
    ]
})

final_results
